In [ ]:
import numpy as np              # numerical computations
import pandas as pd             # data manipulation and analysis
import matplotlib.pyplot as plt # plotting and visualization
import statsmodels.api as sm    # statistical modeling
import matplotlib.image as img
import seaborn as sns
import plotly.express as px

In [ ]:
data = pd.read_csv("/content/advertising.csv")


In [ ]:
X = data[["TV", "Radio", "Newspaper"]]
Y = data["Sales"]


In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, Y,
    test_size=0.2,      # 20% for testing, 80% for training
    random_state=42     # ensures reproducibility - same split every time you run it
)

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

# Train on SCALED data
model = LinearRegression()
model.fit(X_train, y_train)

# Predict on SCALED test data
y_pred = model.predict(X_test)

# Evaluate (same as before — y is not scaled, so metrics stay interpretable)
mse = mean_squared_error(y_test, y_pred)
rmse = mse ** 0.5
mae = mean_absolute_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

print(f"MSE:  {mse:.2f}")
print(f"RMSE: {rmse:.2f}")
print(f"MAE:  {mae:.2f}")
print(f"R²:   {r2:.4f}")


# **First look at the data**

In [ ]:
data.head()  # View first few rows

In [ ]:
data.shape  # rows, columns: Does row/column count match what you expected from the source?


In [ ]:
data.sample(5) #Does sample() show anything head() didn't — weird values, different pattern

In [ ]:
data.tail()  # last 5 rows:Do head() and tail() look consistent, or does the data change character partway


# **Structure & types**

In [ ]:
data.info()  # Check data types and missing values
# Look for:
# Wrong dtypes: numbers stored as object (string) — usually means hidden characters like commas, currency symbols, or stray text mixed into a numeric column
# Dates stored as object instead of datetime
# Non-Null Count column — compare it to total rows to spot missing data before you even run isnull()
# Unexpectedly high memory usage — signals you may need category dtype for repetitive strings

In [ ]:
data.dtypes

# **Statistical summary**

In [ ]:
# Summary statistics
data.describe()
# Look for:
# min/max that don't make sense — negative age, 200-year-old customer, 0 as a price
# mean vs. median (50%) far apart → skewed distribution or outliers
# std very small → column might be near-constant (low information, candidate to drop)
# std very large relative to mean → high variance, may need scaling or transformation
# For describe(include='object'): check unique — is a column supposed to be categorical but has way more unique values than expected (messy/inconsistent text)?

In [ ]:
# Include all columns
data.describe(include='all')

# **Missing values**

In [ ]:
data.isnull().sum()         # count of nulls per column (not df.null())
# Look for:
# Which columns have missing data and how much (a column 90% missing is a drop candidate; 2% missing is an easy impute)
# Missing values in your target column — you almost always need to drop those rows, since you can't train on an unknown label

In [ ]:
data.isnull().sum() / len(data) * 100      # percentage missing — more useful


**Duplicates**

In [ ]:
data.duplicated().sum()                  # count of duplicate rows (not df.duplicate())


In [ ]:
data['TV'].duplicated().sum()

In [ ]:
data[data.duplicated()]                    # actually view the duplicate rows


In [ ]:
data.drop_duplicates(inplace=True)

# **Unique values / cardinality (important, people often skip this)**
Look for:
Inconsistent categories: "USA", "U.S.A", "United States" all meaning the same thing — needs cleaning
High cardinality: a categorical column with thousands of unique values (like free-text or IDs) — one-hot encoding it would blow up your feature space
Typos/rare categories: value_counts() showing one category with 5,000 rows and another with 1 row — likely a typo or needs grouping into "Other"
Columns that are supposed to be categorical but show unexpected values (e.g., a "gender" column with 15 unique values)

In [ ]:
data.nunique()                           # unique values per column
             # list unique values

In [ ]:
data['TV'].value_counts()   # frequency of each category


In [ ]:
data['TV'].unique()             # list unique values

# **Target variable check (for supervised learning)**
Class imbalance — if one class is 95% and the other 5%, plain accuracy will mislead you; you'll need stratified splits, resampling, or class weights
For regression targets: check distribution shape (skewed target may benefit from log-transform)

In [ ]:
data['Sales'].value_counts()                    # class balance


# **Correlation (numeric relationships)**
Look for:

Multicollinearity: two features correlated with each other at ~0.9+ → redundant, consider dropping one (especially matters for linear models)

Weak correlation with target across the board → might mean you need better feature engineering, not more data

Careful: corr() only catches linear relationships — a feature can be very predictive in a non-linear way and still show near-zero correlation

In [ ]:
data.corr()  # correlation matrix



In [ ]:
data.corr()['Sales'].sort_values(ascending=False)   # correlation with target specifically

# **Outlier check**

In [ ]:
data.boxplot()                # visual


In [ ]:
# data[(data['Newspaper'] - data['Newspaper'].mean()).abs() > 3*data['Newspaper'].std()]   # z-score style check

In [ ]:
Q1 = data['Newspaper'].quantile(0.25)
Q3 = data['Newspaper'].quantile(0.75)
IQR = Q3 - Q1

lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

# View outliers first (don't skip this)
outliers = data[(data['Newspaper'] < lower_bound) | (data['Newspaper'] > upper_bound)]
print(outliers)



In [ ]:
# Drop them
data = data[(data['Newspaper'] >= lower_bound) & (data['Newspaper'] <= upper_bound)]

In [ ]:
# # Visualize relationships between each ad channel and Sales:
# plt.scatter(data["TV"], data["Sales"], alpha=0.9)
# # the alpha parameter controls the transparency of the plotted markers.
# # alpha = 0: Completely transparent (invisible). alpha = 1: opaque (solid).
# plt.xlabel("TV Ad Spend")
# plt.ylabel("Sales")
# plt.title("TV Ad Spend vs Sales")
# plt.show()

In [ ]:
# # Visualize relationships between each ad channel and Sales:
# plt.scatter(data["Radio"], data["Sales"], alpha=0.5)
# # the alpha parameter controls the transparency of the plotted markers.
# # alpha = 0: Completely transparent (invisible). alpha = 1: opaque (solid).
# plt.xlabel("Radio Ad Spend")
# plt.ylabel("Sales")
# plt.title("Radio Ad Spend vs Sales")
# plt.show()

In [ ]:
# # Visualize relationships between each ad channel and Sales:
# plt.scatter(data["Newspaper"], data["Sales"], alpha=0.5)
# # the alpha parameter controls the transparency of the plotted markers.
# # alpha = 0: Completely transparent (invisible). alpha = 1: opaque (solid).
# plt.xlabel("Newspaper Ad Spend")
# plt.ylabel("Sales")
# plt.title("Newspaper Ad Spend vs Sales")
# plt.show()

In [ ]:
# import seaborn as sns

In [ ]:
# # distribution plot
# sns.distplot(data['TV'])

In [ ]:
# sns.distplot(data['Radio'])

In [ ]:
# # The modern way to write your exact code:
# sns.histplot(data=data, x='Radio', kde=True)

In [ ]:
# sns.displot(data['Newspaper'])

In [ ]:
# q1 = data['Sales'].quantile(0.25)
# q3 = data['Sales'].quantile(0.75)
# iqr = q3 - q1
# outlier_min = q1 - 1.5 * iqr
# outlier_max = q3 + 1.5 * iqr

In [ ]:
# def find_outlier(df):
#     q1=df.quantile(0.25)
#     q3=df.quantile(0.75)
#     iqr=q3-q1
#     lower=q1-1.5*iqr
#     upper=q3+1.5*iqr
#     outtliers= df[(df<lower)|(df>upper)]
#     return outtliers

In [ ]:
# print(find_outlier(data['Sales']))
# print(find_outlier(data['TV']))
# print(find_outlier(data['Radio']))


In [ ]:
# find_outlier(data['Newspaper'])

In [ ]:
# import plotly.express as px
# px.box(data,x='Newspaper')

In [ ]:
# def drop_outliers(df):
#     q1=df.quantile(0.25)
#     q3=df.quantile(0.75)
#     iqr=q3-q1
#     lower=q1-1.5*iqr
#     upper=q3+1.5*iqr
#     outtliers= df[~((df<lower)|(df>upper)).any(axis=1)]
#     drop_outlier = outtliers.dropna().reset_index()
#     return drop_outlier

In [ ]:
# data=drop_outliers(data)

In [ ]:
# plt.figure(figsize=(15,10))

# for i , column in enumerate(data.columns , 1):
#     plt.subplot(4, 5 , i)
#     sns.boxplot(y=data[column])
#     plt.title(f'Boxplot of {column}')

# plt.tight_layout()
# plt.show()

In [ ]:
# Predict Sales using TV, Radio, and Newspaper ad spends.ad channel and Sales
X = data[["TV", "Radio", "Newspaper"]]

In [ ]:
X

In [ ]:
Y = data["Sales"]
Y

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, Y,
    test_size=0.2,      # 20% for testing, 80% for training
    random_state=42     # ensures reproducibility - same split every time you run it
)

In [ ]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)   # fit + transform on train
X_test_scaled = scaler.transform(X_test)          # transform ONLY on test (no fit!)

In [ ]:
import pandas as pd
print(X_train.corr())

In [ ]:
print(f"X_train: {X_train.shape}, X_test: {X_test.shape}")
print(f"y_train: {y_train.shape}, y_test: {y_test.shape}")

In [ ]:
from sklearn.linear_model import LinearRegression

# Train on SCALED data
model = LinearRegression()
model.fit(X_train_scaled, y_train)

# Predict on SCALED test data
y_pred = model.predict(X_test_scaled)




In [ ]:

# Evaluate (same as before — y is not scaled, so metrics stay interpretable)
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

mse = mean_squared_error(y_test, y_pred)
rmse = mse ** 0.5
mae = mean_absolute_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

print(f"MSE:  {mse:.2f}")
print(f"RMSE: {rmse:.2f}")
print(f"MAE:  {mae:.2f}")
print(f"R²:   {r2:.4f}")

In [ ]:
print(f"Sales mean: {y_test.mean():.2f}")
print(f"Sales std: {y_test.std():.2f}")
print(f"Sales range: {y_test.min()} to {y_test.max()}")